# HotelDataAgentRuntimePublish

Validate the draft HotelOperationsAgent definition and optionally publish the staging configuration
through the Fabric DataAgent runtime API.

In [ ]:
workspace_id = "00000000-0000-0000-0000-000000000000"
data_agent_id = "00000000-0000-0000-0000-000000000000"
capacity_id = "00000000-0000-0000-0000-000000000000"
publish_mode = "validate_only"
published_description = "HotelOperationsAgent draft publication"

In [ ]:
from __future__ import annotations

import json
import time
import sempy.fabric as fabric

client = fabric.FabricRestClient()
base = f"/v1/workspaces/{workspace_id}/dataAgents/{data_agent_id}"

def await_operation(response, operation_name):
    if response.status_code == 200:
        return response
    if response.status_code != 202:
        raise RuntimeError(f"{operation_name} failed: {response.status_code} {response.text[:500]}")
    location = response.headers.get("Location") or response.headers.get("location")
    if not location:
        raise RuntimeError(f"{operation_name} returned 202 without a Location header")
    retry_after = int(response.headers.get("Retry-After", "5"))
    for _ in range(60):
        time.sleep(max(1, retry_after))
        poll = client.get(location)
        if poll.status_code != 200:
            raise RuntimeError(f"{operation_name} poll failed: {poll.status_code} {poll.text[:500]}")
        payload = poll.json()
        status = str(payload.get("status", "")).lower()
        if status in {"succeeded", "completed"}:
            result_response = client.get(location.rstrip("/") + "/result")
            return result_response if result_response.status_code == 200 else poll
        if status in {"failed", "cancelled"}:
            raise RuntimeError(f"{operation_name} failed: {json.dumps(payload, default=str)}")
        retry_after = int(poll.headers.get("Retry-After", retry_after))
    raise TimeoutError(f"{operation_name} did not complete within the polling limit")

definition = client.post(base + "/getDefinition", json={"format": "JSON"})
definition = await_operation(definition, "DataAgent getDefinition")
result = {"workspace_id": workspace_id, "data_agent_id": data_agent_id, "capacity_id": capacity_id, "publish_mode": publish_mode, "published_description": published_description, "validation": {"definition_status_code": definition.status_code}}
if publish_mode == "deploy":
    publish = client.post(base + "/staging/publish", json={"publishedDescription": published_description})
    publish = await_operation(publish, "DataAgent publish")
    result["publish"] = publish.json()

In [ ]:
notebookutils.notebook.exit(json.dumps(result, default=str))